# Address Cleaning

In [ ]:
from pyspark.sql import SparkSession

# Create SparkSession
spark = SparkSession.builder.master("local").appName("test").getOrCreate()

# Read CSV file
df = spark.read.csv(r"test_files/test_1.csv")

## Cleaning Examples

### Clean Punctuation

Cleans up punctuation from address strings by removing or fixing unwanted characters, while preserving hyphens and periods where necessary (e.g., between numbers or block names).

In [ ]:
from address_toolkit.cleaning import clean_punctuation

cleaned_punctation_df = clean_punctuation(df, '_c1', create_flag = True, overwrite = True)
cleaned_punctation_df.filter(cleaned_punctation_df.punctuation_cleaned_flag == 1).show(truncate = False)

### Denoise Addresses

Removes noise words from the input address column. Noise words are defined as those matching the `noise_pattern` regex parameter. By default, this will be letters appearing 3 or more times consecutively e.g "AAAA"

In [ ]:
from address_toolkit.cleaning import denoise_addresses

noise_removed_df = denoise_addresses(df, '_c1', create_flag=True, overwrite=True)
noise_removed_df.filter(noise_removed_df.noise_removed_flag == 1).show(truncate = False)

### Deduplicate Addresses

Deduplicates addresses inter and intracomponently. If `intercomponents = True`, the addresses will drop duplicates between components i.e. `123 Main Road, 123 Main Road, London` -> `123 Main Road, London`. If `intracomponents = True`, the address will drop duplicates within components i.e. `123 Main Road, London London` -> `123 Main Road, London`. The `tolerance` parameter will only drop duplicates with length over the integer passed to it.

In [ ]:
from address_toolkit.cleaning import deduplicate_addresses

deduplicated_df = deduplicate_addresses(df, "_c1", intracomponents = False, intercomponents = True, tolerance = 0, create_flag = True, overwrite = True)
deduplicated_df.filter(deduplicated_df.deduplicated_address_flag == 1).show(truncate = False)

### Deduplicate Postcodes

Detects and removes duplicated patterns across both standard and irregular formatted postcodes

In [ ]:
from address_toolkit.cleaning import deduplicate_postcodes

deduplicated_postcodes_df = deduplicate_postcodes(df, "_c1", create_flag = True, overwrite = True)
deduplicated_postcodes_df.filter(deduplicated_postcodes_df.postcode_deduplicated_flag == 1).show(truncate = False)

### Rectify Postcodes

Corrects UK postcodes with incorrect spacing i.e. `S M6 1E E -> SM6 1EE`. This function seeks to perform componently, therefore poorly formatted addresses are a limiting factor in its performance.


In [ ]:
from address_toolkit.cleaning import rectify_postcodes

rectified_postcodes_df = rectify_postcodes(df, '_c1', create_flag = True, overwrite = True)
rectified_postcodes_df.filter(rectified_postcodes_df.rectified_postcode_flag == 1).show(truncate = False)

### Standardise Street Types 

Standardises street type abbreviations and common misspellings within an address column, applying a set of predefined rules to replace short forms like `Str` with `Street` and fix common typos.

In [ ]:
from address_toolkit.cleaning import standardise_street_types

standardised_street_types_df = standardise_street_types(df, '_c1', create_flag = True, overwrite = True)
standardised_street_types_df.filter(standardised_street_types_df.street_type_standardised_flag == 1).show(truncate = False)

### Prettify Addresses

Ensures addresses are in title case with postcodes formatted in uppercase.

In [ ]:
from address_toolkit.cleaning import prettify_addresses

prettified_addresses_df = prettify_addresses(df, '_c1', overwrite = True)
prettified_addresses_df.show(truncate = False)

## Validating Examples

### Validate Address Components

Flags addresses with components matching those in a `component_list` to the degree of specified in `similarity_threshold`, leveraging fuzzy matching techniques. Available comparator lists can either be set by the user, or imported from `address_toolkit.resources`. 

Some of the available predefined comparator lists include `town_list`, `city_list`, `place_list`, `county_list`, `suburb_list`.

Validating components against the `town_list` imported from `address_toolkit.resources`. This creates a flag indicating whether there are components in the address column matching any of those in the `town_list` to the degree specified in `similarity_threshold`.

In [ ]:
from address_toolkit.validating import validate_from_list
from address_toolkit.resources import town_list

validate_components_df = validate_from_list(df, '_c1', component_name = 'town', component_list=town_list, similarity_threshold=50, scores = True)
validate_components_df = validate_components_df.show(truncate = False)

Validating components against the `county_list` imported from `address_toolkit.resources`. This creates a flag indicating whether there are components in the address column matching any of those in the `county_list` to the degree specified in `similarity_threshold`.

In [ ]:
from address_toolkit.resources import county_list
validate_components_df = validate_from_list(df, '_c1', component_name = 'county', component_list=county_list, similarity_threshold=90, scores = False)
validate_components_df = validate_components_df.show(truncate = False)

Validating components against the `disallowed_country_list` imported from `address_toolkit.resources`. This creates a flag indicating whether there are components in the address column matching any of those in the `disallowed_country_list` to the degree specified in `similarity_threshold`.

In [ ]:
from address_toolkit.resources import disallowed_country_list

validate_components_df = validate_from_list(df, '_c1', component_name = 'disallowed_country', component_list=disallowed_country_list, similarity_threshold=90, scores = False)
validate_components_df = validate_components_df.show(truncate = False)

### Validate Postcodes

Flags addresses with an invalid postcode format, including those with "ZZ99" and those with forbidden characters in the inner part of the postcode.

In [ ]:
from address_toolkit.validating import validate_postcodes

valid_postcode_df = validate_postcodes(df, '_c1')
valid_postcode_df = valid_postcode_df.filter(valid_postcode_df.validated_postcode_flag == 1).show(truncate = False)

### Validate from RegEx

Flags addresses containing a match from a given regex.

In [ ]:
from address_toolkit.validating import validate_from_regex
from address_toolkit.resources import floor_regex

validate_regex_df = validate_from_regex(df, "_c1", component_name = 'floor', regex_pattern = floor_regex)
validate_regex_df = validate_regex_df.filter(validate_regex_df.validated_floor_flag == 1).show(truncate = False)

## Extracting Examples

### Extract Postcodes

Extracts postcodes from an address string. If `replace` is toggled to `True`, the postcode will be replaced in the address string.

In [ ]:
from address_toolkit.extracting import extract_postcodes

extracted_postcode_df = extract_postcodes(df, '_c1', replace = False)
extracted_postcode_df = extracted_postcode_df.show(truncate = False)

### Extract Components From List

Extracts address components that match to a `component_list` to the degree of similarity specified in `similarity_threshold`.

There are a range of component lists available from `address_toolkit.resources` such as towns, villages and counties.

In [ ]:
from address_toolkit.extracting import extract_components_from_list
from address_toolkit.resources import town_list

extracted_town_df = extract_components_from_list(df, '_c1', component_name = 'town', component_list = town_list, similarity_threshold = 90, scores = True, replace = False)
extracted_town_df = extracted_town_df.show(truncate = False)

### Extract Components from Regex

Extracts components which match to a regex pattern passed. Some predefined regex patterns have been defined in `address_toolkit.resources` such as `flat_regex`, `room_regex`, `unit_regex`.

In [ ]:
from address_toolkit.extracting import extract_components_from_regex
from address_toolkit.resources import floor_regex

extracted_room_df = extract_components_from_regex(df, "_c1", component_name = 'floor', regex_pattern = floor_regex, replace = False)
extracted_room_df = extracted_room_df.filter(extracted_room_df.floor != "").show(truncate = False)

## Contextualising Examples

Contextualises address components with borough/district and county. Contextualisations will only occur if the postcode also corresponds to that from the component resource lookup, this is a necessary safeguard to prevent faulty imputations.

In [ ]:
from address_toolkit.contextualising import contextualise_from_lookup
from address_toolkit.resources import town_lookup

contextualised_towns_df = contextualise_from_lookup(df, '_c1', component_name = 'town', similarity_threshold = 100, component_lookup = town_lookup, overwrite = True)
contextualised_towns_df.filter(contextualised_towns_df.contextualised_from_town_flag == 1).show(truncate = False)

## Workflows Examples

### Clean Addresses

Cleans address data by applying a series of cleaning functions including punctuation cleaning,
denoising, deduplication of addresses and postcodes, postcode rectification, and street type standardization.

In [ ]:
from address_toolkit.workflows import clean_addresses

cleaned_df = clean_addresses(df, '_c1', create_flag = True)
cleaned_df.filter(cleaned_df.deduplicated_address_flag==1).show(truncate = False)

## Validate Addresses

 Validates address data by applying a series of validation functions including address component validation and postcode validation.

Notes:
- Address component validation will be run for towns, cities and counties.

In [ ]:
from address_toolkit.workflows import validate_addresses

validated_df  = validate_addresses(df, '_c1', similarity_threshold = 95)
validated_df.show(truncate = False)

## Extract Address Components

Extracts address components postcodes, towns, cities, and counties from the given address column.

In [ ]:
from address_toolkit.workflows import extract_address_components

extracted_components_df = extract_address_components(df, '_c1', similarity_threshold = 95, replace = True)
extracted_components_df.show(truncate = False)